# 🏰 Deine Daten sind dein Burggraben

Generische Modelle sind Massenware — jeder kann GPT-4o nutzen. **Das Modell ist gemietet, deine Daten gehören dir.** Tuning macht den Unterschied, den kein Konkurrent kopieren kann.

In diesem Notebook nutzen wir die **echten Ticket-Daten** aus dem Projekt (`csv/data.csv`) und zeigen:
1. Generischer Prompt auf Ticket-Daten → meh Score
2. Jetzt tunen wir mit DEINEN echten Ticket-Daten...
3. Viel besser! DEINE Daten machen den Unterschied.

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, pandas as pd, ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task
from dspy_tasks.actions import run_baseline, run_optimization, compare_models
from dspy_tasks.visualize import *

# Show the real ticket data
df = pd.read_csv("../csv/data.csv", encoding="latin-1")
print(f"📊 Real ticket data: {len(df)} tickets")
print(f"Columns: {', '.join(df.columns[:10])}...")
df[["Summary*", "Priority*", "Status*", "Assigned Group*+"]].head(5)

In [ ]:
task = get_task("ticket_routing")
examples = task.load_examples()
print(f"Training data: {len(examples)} ticket examples\n")
for ex in examples[:3]:
    print(f"  Summary: {str(ex.summary)[:80]}...")
    print(f"  → Category: {ex.category} | Priority: {ex.priority} | Team: {ex.assigned_group}\n")

## Generischer Prompt vs. Domain-Tuning

Jetzt wird's spannend: wir lassen den gleichen Task einmal mit einem generischen Prompt und einmal mit automatischem Tuning laufen. Die Daten kommen aus eurem echten Ticket-System!

Erwartung: der generische Prompt liefert ein "geht so" Ergebnis. Nach dem Tuning mit deinen Daten? **Deutlich besser.**

In [ ]:
from dspy_tasks.visualize import diagram_compare

diagram_compare(
    before=[
        {"label": "Generischer Prompt", "detail": "Keine Domain-Daten", "icon": "📝", "color": "#a4262c"},
        {"label": "LLM", "detail": "Gemietetes Modell", "icon": "🤖", "color": "#8a8886"},
        {"label": "Ergebnis", "detail": "Mittelmässig", "icon": "😐", "color": "#a4262c"},
    ],
    after=[
        {"label": "Getuned mit deinen Daten", "detail": "Echte Tickets", "icon": "📊", "color": "#107c10"},
        {"label": "LLM", "detail": "Gleich gemietetes Modell", "icon": "🤖", "color": "#8a8886"},
        {"label": "Ergebnis", "detail": "Deutlich besser!", "icon": "🎯", "color": "#107c10"},
    ],
    title="Vorher vs. Nachher: Deine Daten machen den Unterschied",
)

In [ ]:
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy
MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
btn = run_button("Generic vs. Tuned")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        print(f"⏳ Running ticket routing: generic vs. domain-tuned on {model_dd.value}...")

        result = run_optimization("ticket_routing", model_dd.value, "BootstrapFewShot", max_eval=10)

        display_improvement(result.baseline_score, result.optimized_score)
        display_prompt_diff(result.prompt_before, result.prompt_after,
            title="Generic Prompt vs. Domain-Tuned Prompt")

        display_insight("Der Burggraben",
            f"Ein generischer Prompt erreicht {result.baseline_score:.0%}. "
            f"Getuned mit DEINEN Ticket-Daten: {result.optimized_score:.0%}. "
            "Das Modell ist gemietet, deine Daten gehören dir. "
            "Dieses Tuning ist DEIN Wettbewerbsvorteil — kein Konkurrent kann das kopieren.")

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

In [ ]:
compare_btn = run_button("Compare Models on Domain Tasks")
compare_out = widgets.Output()

def on_compare(b):
    with compare_out:
        compare_out.clear_output()
        result = compare_models("ticket_routing", MODELS, max_eval=8)

        model_scores = {}
        for m in MODELS:
            short = m.split("/")[-1]
            model_scores[short] = {
                "baseline": result.baseline_scores[m],
                "optimized": result.optimized_scores[m],
            }
        fig = bar_comparison("Ticket Routing: Model Comparison", model_scores)
        fig.show()

        # Check if small model + tuning beats big model
        if len(MODELS) >= 2:
            small_opt = result.optimized_scores.get(MODELS[1], 0)
            big_base = result.baseline_scores.get(MODELS[0], 0)
            if small_opt > big_base:
                display_insight("💰 Der Kosten-Insight",
                    f"{MODELS[1].split('/')[-1]} optimiert ({small_opt:.0%}) schlägt "
                    f"{MODELS[0].split('/')[-1]} unoptimiert ({big_base:.0%})! "
                    "Ein getuntes kleines Modell schlägt ein ungetuntes grosses — und kostet 10x weniger.")

compare_btn.on_click(on_compare)
display(compare_btn, compare_out)

In [ ]:
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task("comparative_analysis"), get_task("instruction_constraints")]],
    description="Task:")
run_btn = run_button("Run & Optimize")
run_out = widgets.Output()

def on_run_extra(b):
    with run_out:
        run_out.clear_output()
        result = run_optimization(task_dd.value, model_dd.value, max_eval=8)
        display_improvement(result.baseline_score, result.optimized_score)
        display_results_table(result.individual_scores if hasattr(result, 'individual_scores') else [])

run_btn.on_click(on_run_extra)
display(widgets.HBox([task_dd, run_btn]), run_out)

## ⏭️ Weiter geht's!

**Das Modell ist gemietet. Die Daten gehören dir. Das Tuning ist dein Engineering.**

Deine Daten + automatisches Tuning = ein Burggraben, den kein Konkurrent kopieren kann. Und das Beste: ein getuntes kleines Modell schlägt oft ein ungetuntes grosses — und kostet 10x weniger.

Aber können auch **Agenten** optimiert werden? Agenten, die Tools nutzen und Entscheidungen treffen? Das ist Notebook 06!